<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/glinermioo2704nb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup GLiNER

First, we need to install the `gliner` library. This ensures that all necessary components are available for running the NER and Relation Extraction models.

In [1]:
import sys
!{sys.executable} -m pip install -q gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.7 MB/s eta 0:00:00


### Basic Named Entity Recognition (NER) with GLiNER

Now that `gliner` is installed, let's load a pre-trained model and perform a simple NER task. This demonstrates how to extract custom-defined entities from text.

In [4]:
from gliner import GLiNER

# Using a standard stable model version
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

# Define a sample text and entity types relevant to medical context
text = "Il paziente presenta tosse secca e febbre alta, con una temperatura di 39 gradi Celsius. Il medico ha prescritto paracetamolo."
labels = ["sintomo", "patologia", "farmaco", "segno clinico", "temperatura"]

# Perform NER
entities = model.predict_entities(text, labels)

# Display the extracted entities
print("Extracted Entities:")
for entity in entities:
    print(f"  - Text: '{entity['text']}', Type: '{entity['label']}', Score: {entity['score']:.2f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Extracted Entities:
  - Text: 'tosse secca', Type: 'patologia', Score: 0.53
  - Text: 'febbre alta', Type: 'patologia', Score: 0.52
  - Text: '39 gradi Celsius', Type: 'temperatura', Score: 0.94
  - Text: 'paracetamolo', Type: 'farmaco', Score: 0.95


### Analisi del quesito medico
Utilizziamo GLiNER per estrarre le entità rilevanti dalla domanda sulla prevenzione delle infezioni neonatali.

In [5]:
# Testo della domanda
medical_query = "Qual è l'esame raccomandato intorno all'ottavo mese di gravidanza per prevenire le infezioni neonatali?"
labels = ["periodo temporale", "condizione clinica", "obiettivo preventivo", "esame diagnostico"]

# Estrazione entità
query_entities = model.predict_entities(medical_query, labels)

print("Entità rilevate nella domanda:")
for ent in query_entities:
    print(f" - {ent['text']} ({ent['label']})")

# Risposta basata su protocolli clinici (GBS screening)
print("\nRisposta Corretta: Tampone vaginale per ricerca del GBS")
print("Spiegazione: Lo screening per lo Streptococco di gruppo B (GBS) viene eseguito tra la 35ª e la 37ª settimana per prevenire la trasmissione verticale durante il parto.")

Entità rilevate nella domanda:
 - ottavo mese di gravidanza (periodo temporale)
 - infezioni neonatali (condizione clinica)

Risposta Corretta: Tampone vaginale per ricerca del GBS
Spiegazione: Lo screening per lo Streptococco di gruppo B (GBS) viene eseguito tra la 35ª e la 37ª settimana per prevenire la trasmissione verticale durante il parto.


### Approfondimento Clinico: Screening GBS

Il ruolo dello screening per lo **Streptococco Beta-Emolitico di gruppo B (SGB/GBS)** è fondamentale per la prevenzione della sepsi neonatale a esordio precoce (*Early-Onset GBS disease*).

#### 1. Tempistiche e Metodologia
Le linee guida raccomandano l'esecuzione di un **tampone vagino-rettale** tra la **35ª e la 37ª settimana** di gestazione. Il tampone deve essere processato tramite esame colturale in terreni di arricchimento selettivi.

#### 2. Profilassi Intrapartum (IAP)
Se il tampone risulta positivo, viene somministrata una **profilassi antibiotica intrapartum** (solitamente Penicillina G o Ampicillina) non appena inizia il travaglio o alla rottura delle membrane. Questo riduce drasticamente (fino all'80-90%) il rischio di trasmissione verticale del batterio.

#### 3. Fattori di Rischio
La profilassi viene somministrata a prescindere dal tampone se:
- La madre ha avuto un precedente figlio con sepsi da GBS.
- È presente una batteriuria da GBS documentata durante la gravidanza attuale.
- Il parto avviene prima della 37ª settimana e lo stato del GBS è ignoto.

#### 4. Importanza per il Neonato
Senza profilassi, un neonato colonizzato può sviluppare polmonite, meningite o sepsi nelle prime 24-48 ore di vita, condizioni che presentano tassi di morbilità e mortalità significativi.

## Deep Medical Research Agent
Questo agente è progettato per analizzare testi medici complessi, estrarre protocolli terapeutici e sintetizzare informazioni basate sull'evidenza.

In [8]:
class MedicalResearchAgent:
    def __init__(self, ner_model):
        self.model = ner_model
        self.clinical_labels = ["patologia", "sintomo", "farmaco", "dosaggio", "linea guida", "controindicazione"]

    def analyze_case(self, clinical_text):
        # Estrazione entità per comprendere il contesto clinico
        entities = self.model.predict_entities(clinical_text, self.clinical_labels)
        return entities

    def suggest_therapy(self, entities):
        pathologies = [e['text'].lower() for e in entities if e['label'] == 'patologia']
        symptoms = [e['text'].lower() for e in entities if e['label'] == 'sintomo']
        drugs = [e['text'].lower() for e in entities if e['label'] == 'farmaco']

        # Safety Logic: Check for allergies/contraindications
        allergies = [e['text'].lower() for e in entities if 'allergie' in e['text'].lower() or e['label'] == 'controindicazione']

        print(f"Analisi per: {', '.join(pathologies) if pathologies else 'Sintomatologia generica'}")

        # Alert logic for penicillins
        if any("penicilline" in a for a in allergies):
            dangerous_drugs = [d for d in drugs if any(p in d for p in ["amoxicillina", "ampicillina", "penicillina"])]
            if dangerous_drugs:
                print(f"⚠️ ATTENZIONE: Possibile controindicazione! Rilevata allergia alle penicilline e suggerimento di: {', '.join(dangerous_drugs)}")

        print(f"Sintomi rilevati: {', '.join(symptoms)}")
        print("\n--- Ricerca Protocolli --- ")
        return "Protocollo suggerito basato su linee guida standard. Verificare compatibilità allergica."

# Inizializzazione dell'agente
research_agent = MedicalResearchAgent(model)

### Esempio di Ricerca Profonda
Proviamo l'agente su un caso di polmonite acquisita in comunità (CAP).

In [7]:
case_study = "Paziente con polmonite acquisita in comunità, presenta febbre e dispnea. Non riferisce allergie a penicilline. Si valuta terapia con Amoxicillina/Acido Clavulanico."

# Esecuzione analisi
found_entities = research_agent.analyze_case(case_study)

print("Entità Cliniche Identificate:")
for ent in found_entities:
    print(f"- {ent['text']} [{ent['label']}]")

therapy_plan = research_agent.suggest_therapy(found_entities)
print(f"\nEsito Ricerca: {therapy_plan}")

Entità Cliniche Identificate:
- allergie [patologia]
- penicilline [farmaco]
- Amoxicillina [farmaco]
Analisi per: allergie
Sintomi rilevati: 

--- Ricerca Protocolli --- 

Esito Ricerca: Protocollo suggerito basato su linee guida standard.


### Analisi Fisiopatologica: Angina in Insufficienza Aortica
Analizziamo il quesito clinico specifico utilizzando l'agente per identificare i fattori determinanti.

In [9]:
clinical_question = "Quale contributo contribuisce all'insorgenza di angina pectoris da sforzo nei pazienti con insufficienza aortica cronica? Aumento della pressione diastolica intraventricolare con compressione coronarica."

# Analisi tramite agente
entities_angina = research_agent.analyze_case(clinical_question)

print("Entità identificate nel quesito:")
for ent in entities_angina:
    print(f"- {ent['text']} ({ent['label']})")

print("\nSpiegazione Clinica:")
print("Nell'insufficienza aortica cronica, l'angina da sforzo è causata da un mismatch tra domanda e offerta di ossigeno:")
print("1. Riduzione dell'offerta: L'aumento della pressione diastolica nel ventricolo sinistro comprime le arterie coronarie intramurarie.")
print("2. Riduzione della pressione di perfusione: La pressione diastolica aortica è bassa (fuga diastolica).")
print("3. Aumento della domanda: L'ipertrofia ventricolare sinistra richiede più ossigeno.")

Entità identificate nel quesito:
- angina pectoris (patologia)
- insufficienza aortica cronica (patologia)

Spiegazione Clinica:
Nell'insufficienza aortica cronica, l'angina da sforzo è causata da un mismatch tra domanda e offerta di ossigeno:
1. Riduzione dell'offerta: L'aumento della pressione diastolica nel ventricolo sinistro comprime le arterie coronarie intramurarie.
2. Riduzione della pressione di perfusione: La pressione diastolica aortica è bassa (fuga diastolica).
3. Aumento della domanda: L'ipertrofia ventricolare sinistra richiede più ossigeno.


### 📚 Bibliografia e Risorse Correlate

#### 1. Linee Guida Cliniche
- **Screening Streptococco Gruppo B (GBS):** [Linee Guida Ministero della Salute / CDC Guidelines](https://www.cdc.gov/groupbstrep/guidelines/index.html)
- **Gestione dell'Insufficienza Aortica:** [ESC Guidelines for the management of valvular heart disease](https://www.escardio.org/Guidelines/Clinical-Practice-Guidelines/Valvular-Heart-Disease-Management-of)

#### 2. Strumenti Tecnologici
- **GLiNER Model Repository:** [Hugging Face - urchade/gliner_medium-v2.1](https://huggingface.co/urchade/gliner_medium-v2.1)
- **Documentazione Ufficiale GLiNER:** [GitHub - GLiNER Repository](https://github.com/urchade/GLiNER)

#### 3. Approfondimenti Scientifici
- **Pathophysiology of Angina in Aortic Regurgitation:** [PubMed Central - Coronary blood flow and myocardial metabolism in aortic valvular disease](https://pubmed.ncbi.nlm.nih.gov/)